# 01 — Exploratory Data Analysis

**Course:** ADS504 — Machine Learning and Deep Learning for Data Science  
**Dataset:** [Bank Marketing (UCI)](https://archive.ics.uci.edu/dataset/222/bank+marketing)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 40)
sns.set_style('whitegrid')


## Load data


In [ ]:
DATA_PATH = '../data/raw/bank-full.csv'
df = pd.read_csv(DATA_PATH, sep=';')
print('Shape:', df.shape)
df.head()


## Data overview


In [ ]:
df.info()
print('Duplicate rows:', df.duplicated().sum())
print('Null cells per column:')
print(df.isnull().sum())


In [ ]:
numeric_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
target = 'y'

df[numeric_cols].describe().T.round(1)


## Target variable


In [ ]:
df['y_binary'] = (df['y'] == 'yes').astype(int)
base_rate = df['y_binary'].mean() * 100

vc = df['y'].value_counts()
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(vc.index, vc.values, color=['steelblue', 'coral'])
ax.set_title('Target distribution (y)')
ax.set_ylabel('count')
for i, v in enumerate(vc.values):
    ax.text(i, v + 400, f'{v/len(df)*100:.1f}%', ha='center')
plt.tight_layout()
plt.show()

print(f'Yes rate: {base_rate:.1f}%')


The target is imbalanced (about 12% yes). Accuracy alone will be misleading later; use recall, F1, and ROC-AUC.


## Missing and coded values


In [ ]:
missing = {c: (df[c] == 'unknown').mean() * 100 for c in categorical_cols if (df[c] == 'unknown').any()}
missing['pdays == -1'] = (df['pdays'] == -1).mean() * 100
miss_s = pd.Series(missing).sort_values(ascending=False)
print(miss_s.round(1))

print('pdays == -1:', (df['pdays'] == -1).sum())
print('poutcome == unknown:', (df['poutcome'] == 'unknown').sum())
print('previous == 0:', (df['previous'] == 0).sum())


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
miss_s.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('percent of rows')
ax.set_title('Unknown or sentinel-coded values')
plt.tight_layout()
plt.show()


Most unknown values are structural (no prior contact), not random missing data. Keep unknown as its own category in preprocessing.


## Numeric distributions


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, col in zip(axes.ravel(), numeric_cols):
    ax.hist(df[col], bins=50, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel(col)
plt.tight_layout()
plt.show()

print(df[numeric_cols].skew().round(2))


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 4))
for ax, col in zip(axes, ['balance', 'duration', 'campaign', 'previous']):
    ax.boxplot(df[col])
    ax.set_title(col)
plt.tight_layout()
plt.show()


Several numeric features are right-skewed. Outliers look domain-valid, so we will transform rather than drop rows.


## Correlations


In [ ]:
corr = df[numeric_cols + ['y_binary']].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Numeric correlations')
plt.tight_layout()
plt.show()


Duration has the strongest correlation with the target. It is only known after a call, so drop it before modeling.


## Duration and leakage


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for lab, color in [('no', 'steelblue'), ('yes', 'coral')]:
    ax[0].hist(df.loc[df.y == lab, 'duration'], bins=60, alpha=0.6, color=color, label=lab, density=True)
ax[0].set_xlim(0, 1500)
ax[0].set_xlabel('duration (sec)')
ax[0].legend()

ax[1].boxplot([df.loc[df.y == 'no', 'duration'], df.loc[df.y == 'yes', 'duration']], labels=['no', 'yes'])
ax[1].set_ylabel('duration (sec)')
plt.tight_layout()
plt.show()

print('Correlation with target:', round(df['duration'].corr(df['y_binary']), 3))


## Categorical features vs target


In [ ]:
def subscription_rate(col, order=None):
    t = df.groupby(col)['y_binary'].agg(['mean', 'count'])
    if order is not None:
        t = t.reindex(order)
    t['rate_pct'] = t['mean'] * 100
    return t.sort_values('rate_pct', ascending=False)

for col in ['job', 'education', 'marital', 'housing', 'loan', 'contact', 'poutcome']:
    print(f'\n{col}:')
    print(subscription_rate(col).head(8).round(1))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

job_rates = subscription_rate('job').sort_values('rate_pct')
axes[0].barh(job_rates.index.astype(str), job_rates['rate_pct'], color='steelblue')
axes[0].axvline(base_rate, color='red', linestyle='--', linewidth=1)
axes[0].set_title('Subscription rate by job')
axes[0].set_xlabel('rate (%)')

edu_order = ['primary', 'secondary', 'tertiary', 'unknown']
edu_rates = subscription_rate('education', order=edu_order)
axes[1].bar(edu_rates.index.astype(str), edu_rates['rate_pct'], color='steelblue')
axes[1].axhline(base_rate, color='red', linestyle='--', linewidth=1)
axes[1].set_title('Subscription rate by education')
axes[1].set_ylabel('rate (%)')
plt.tight_layout()
plt.show()


## Month: call volume vs conversion


In [ ]:
month_order = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
month_stats = df.groupby('month')['y_binary'].agg(['mean', 'count']).reindex(month_order)
month_stats['rate_pct'] = month_stats['mean'] * 100

fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.bar(month_order, month_stats['count'], color='steelblue', alpha=0.8)
ax1.set_ylabel('call volume')
ax2 = ax1.twinx()
ax2.plot(month_order, month_stats['rate_pct'], color='coral', marker='o')
ax2.axhline(base_rate, color='red', linestyle='--', linewidth=1)
ax2.set_ylabel('subscription rate (%)')
ax1.set_title('Calls and conversion by month')
plt.tight_layout()
plt.show()

month_stats.round(1)


## Cross-tabulations


In [ ]:
pd.crosstab(df['housing'], df['loan'], values=df['y_binary'], aggfunc='mean').round(3) * 100


In [ ]:
pd.crosstab(df['contact'], df['poutcome'], values=df['y_binary'], aggfunc='mean').round(3) * 100


## Age


In [ ]:
df['age_band'] = pd.cut(df['age'], bins=[0, 30, 45, 60, 100], labels=['<=30', '31-45', '46-60', '60+'])
age_rates = subscription_rate('age_band')
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(age_rates.index.astype(str), age_rates['rate_pct'], color='steelblue')
ax.axhline(base_rate, color='red', linestyle='--', linewidth=1)
ax.set_title('Subscription rate by age band')
ax.set_ylabel('rate (%)')
plt.tight_layout()
plt.show()


## Summary

- 45,211 rows, no duplicate rows or NaNs; unknown and pdays=-1 encode prior-contact status.
- Target is imbalanced (~12% yes).
- Strongest categorical signals: poutcome, contact, job, education, month.
- Duration correlates strongly with y but is leakage; exclude from features.
- Numeric features are skewed; plan log transforms and encoding in preprocessing.
